In [ ]:
import os
import face_recognition
import numpy as np
import csv
from scipy.spatial.distance import euclidean

# Paths and threshold
dataset_dir = "dataset"
faces_dir = "faces"
output_csv = "face_matches.csv"
threshold = 0.6

# Load known faces
known_faces = []
for filename in os.listdir(dataset_dir):
    if filename.lower().endswith(('.jpg', '.png')):
        path = os.path.join(dataset_dir, filename)
        image = face_recognition.load_image_file(path)
        encodings = face_recognition.face_encodings(image)
        if encodings:
            name = os.path.splitext(filename)[0]
            known_faces.append((name, encodings[0]))

print(f"{len(known_faces)} faces loaded from the dataset.")

# Match faces and write results
with open(output_csv, mode='w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['image_file', 'matched_name', 'distance'])

    for face_file in os.listdir(faces_dir):
        if face_file.lower().endswith(('.jpg', '.png')):
            face_path = os.path.join(faces_dir, face_file)
            unknown_image = face_recognition.load_image_file(face_path)
            unknown_encodings = face_recognition.face_encodings(unknown_image)

            if not unknown_encodings:
                writer.writerow([face_file, 'No face detected', ''])
                continue

            unknown_embedding = unknown_encodings[0]
            best_match, min_distance = None, float('inf')

            for name, known_emb in known_faces:
                dist = euclidean(unknown_embedding, known_emb)
                if dist < min_distance:
                    min_distance, best_match = dist, name

            if min_distance > threshold:
                best_match = 'Unknown'

            writer.writerow([face_file, best_match, f"{min_distance:.4f}"])
